<a href="https://colab.research.google.com/github/pxs1990/NLP_LLM/blob/main/peft_llm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
! pip install -U transformers datasets peft accelerate scikit-learn pandas torch
! pip install -U bitsandbytes # Optional for low VRAM:

In [ ]:
import os, pandas as pd, numpy as np from datasets
import Dataset, DatasetDict from sklearn.model_selection
import train_test_split from sklearn.metrics
import accuracy_score, f1_score, classification_report
import torch from transformers import (
     AutoTokenizer,
     AutoModelForSequenceClassification,
     DataCollatorWithPadding,
     TrainingArguments,
     Trainer,
     )
from peft import LoraConfig, get_peft_model

In [ ]:
MODEL_NAME = "bert-base-uncased"  # any BERT-like model

In [ ]:
# ========= 1) Load & prepare data =========
df = pd.read_csv("train.csv")  # expects 'text','label'
assert {"text","label"}.issubset(df.columns), "CSV must have text,label columns"

# Encode string labels to 0..N-1 (keeps ints as-is)
if df["label"].dtype == "object":
    df["label"] = df["label"].astype("category")
	label2id = {c: i for i, c in enumerate(sorted(df["label"].unique()))}
    df["label"] = df["label"].map(label2id)
else:
	label2id = {int(l): int(l) for l in sorted(df["label"].unique())}

id2label = {i: str(k) for k, i in label2id.items()}
num_labels = len(id2label)

train_df, val_df = train_test_split(df, test_size=0.1, stratify=df["label"], random_state=42)

ds = DatasetDict({
    "train": Dataset.from_pandas(train_df, preserve_index=False),
    "validation": Dataset.from_pandas(val_df, preserve_index=False),
})


In [ ]:
# ========= 2) Tokenizer =========
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
def preprocess(ex):
	return tokenizer(ex["text"], truncation=True)
keep_cols = ["label"]
ds = ds.map(preprocess, batched=True, remove_columns=[c for c in ds["train"].column_names if c not in keep_cols])

collator = DataCollatorWithPadding(tokenizer=tokenizer)


In [ ]:
# ========= 3) Base model =========
# (Optional low-VRAM: load in 8-bit with bitsandbytes; uncomment below)
# from transformers import BitsAndBytesConfig
# quant_cfg = BitsAndBytesConfig(load_in_8bit=True)
# base_model = AutoModelForSequenceClassification.from_pretrained(
#     MODEL_NAME, num_labels=num_labels, id2label=id2label, label2id={v:k for k,v in id2label.items()},
#     quantization_config=quant_cfg, device_map="auto"
# )

base_model = AutoModelForSequenceClassification.from_pretrained(
	MODEL_NAME,
    num_labels=num_labels,
    id2label=id2label,
    label2id={v: k for k, v in id2label.items()},
)

In [ ]:
# ========= 4) Apply LoRA adapters with PEFT =========
# For BERT, common target module substrings: "query","key","value","dense"
lora_cfg = LoraConfig(
	r=8,
    lora_alpha=16,
    target_modules=["query","key","value","dense"],
    lora_dropout=0.1,
    bias="none",
    task_type="SEQ_CLS",
)
model = get_peft_model(base_model, lora_cfg)
model.print_trainable_parameters()  # sanity check


In [ ]:
# ========= 5) Def Compute Metrics =========
def compute_metrics(p):
	preds = np.argmax(p.predictions, axis=1)
	return {
        "accuracy": accuracy_score(p.label_ids, preds),
        "f1_macro": f1_score(p.label_ids, preds, average="macro"),
	}


In [ ]:
# ========= 6) Def Training args & Trainer =========
args = TrainingArguments(
    output_dir="bert-lora-multiclass",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    learning_rate=2e-4,        	# LoRA can use a higher LR than full fine-tuning
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_strategy="steps",
    logging_steps=50,
    gradient_accumulation_steps=1,
    fp16=torch.cuda.is_available(),   # enable mixed precision if on GPU
    push_to_hub=False,
)

trainer = Trainer(
    model=model,
	args=args,
    train_dataset=ds["train"],
    eval_dataset=ds["validation"],
    tokenizer=tokenizer,
    data_collator=collator,
    compute_metrics=compute_metrics,
)

trainer.train()